# NEU-DET: YOLOv5 vs Faster R-CNN (end-to-end on Colab)

Runtime → Change runtime type → **T4 GPU**. The whole notebook takes roughly 1.5 hours.

Steps: download NEU-DET → build a stratified 70/15/15 split → train YOLOv5s → train Faster R-CNN → score both on the test split with the same evaluator → export `results/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
!git clone https://github.com/nilaydatta1234/surface-defect-detection.git
%cd surface-defect-detection
!pip install -q -e .

## 2. Download NEU-DET from Kaggle

Upload your `kaggle.json` (Kaggle → Settings → API → Create New Token) when prompted.

In [ ]:
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database -p /content/raw --unzip -q
!find /content/raw -maxdepth 3 -type d | head -20

## 3. Build the YOLO-format dataset and fixed split

In [ ]:
!python -m defect_detection.prepare --raw /content/raw --out data/neu-det

## 4. Train YOLOv5s (COCO-pretrained, 640px, 100 epochs)

In [ ]:
!git clone -q https://github.com/ultralytics/yolov5.git
!pip install -q -r yolov5/requirements.txt
!cd yolov5 && python train.py --img 640 --batch 32 --epochs 100 --seed 0 --cache     --data ../data/neu-det/neu-det.yaml --weights yolov5s.pt     --project ../runs/yolo --name neu-det --exist-ok

## 5. Train Faster R-CNN (ResNet-50 FPN v2, COCO-pretrained, 640px, 24 epochs)

In [ ]:
!python -m defect_detection.frcnn --data data/neu-det --epochs 24 --batch-size 8 --out runs/frcnn

## 6. Evaluate both on the held-out test split

In [ ]:
!python -m defect_detection.evaluate --data data/neu-det --split test     --yolo runs/yolo/neu-det/weights/best.pt --yolo-repo yolov5     --frcnn runs/frcnn/best.pt --out results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results.md").read()))
display(Image("results/samples.jpg"))

## 7. Export results (commit these to the repo)

In [ ]:
!cp runs/yolo/neu-det/results.png results/yolo_training_curves.png
!cp runs/yolo/neu-det/confusion_matrix.png results/yolo_confusion_matrix.png
!cp runs/frcnn/log.csv results/frcnn_training_log.csv
!cp data/neu-det/split_manifest.csv results/split_manifest.csv
!zip -qr results.zip results
files.download("results.zip")